# Data Replication and Failover

Database Replication
- serverless replication of data from one account to another
- one command and runs instantaneously
- executed through a task

Replicaion is the process of syncing database ojects from Primary account to one or more Secondary accounts in different regions or even different cloud providers.

Key objects replicated
1. Permanenet and Transient tables
2. Metadata
3. Integrations : API , Storage, network policy

Replication process
1. Primary database
2. Secondary database
3. Refresh Mechanism




# Database Failover
- When one snowflake account is unavailable, another can be promoted to be primary . This can happen across cloud providers or regions.

While replication moves the data, Failover is the action of switching operations to the secondary site during an outage.
Business Continuity Features:
- Failover groups: You group databses adn account objetcs together. If a region goes down, you fail over the entire group to ensure the environment remains functional
- CLient Redirection: Snowflake provides a connection URL. Instead of changing your application code, you simple point the URL to th enew primary site
- Promotion: When you fail over, the secondary database is promoted to Primary. Once the outage is fixed, changes made to secondary account gets shifted to primary account with ease.

Database objects : Only Permanent and Transient Databases can be replicated.
Credit consumption: you are charged for compute and cloud service and also for data transfer acoss regions/clouds
Database shares: You can replicate databases that are being shared, but there are specific stepos to ensure the Share object exists in the secondary region
External Tables : Standard replication deosn't include the physical files in you ext stages, you must manage underlying storage replication yourself


STEPS TO CREATE REPLICATION

1. Enable replication for the account
ALTER ACCOUNT SET REPLICATION_ALLOWED_TO_ACCOUNTS = <org_name>.<target_account_name>;

2. Create a secondary Database
CREATE DATABASE <db_name> as REPLICA OF <org_name>.<source_account_name>.<db_name>;

3. Perform the inital refresh
ALTER DATABASE <db_name> REFRESH;

4. Schedule recurring refreshes ( optional but recommended)

5. COnfigure a failover group( busines critical)
if you want to be able to 'Failover' (promote the secondary to primary), you should use failover groups. This bundle multiple databases and account objects.


# Secure Data Sharing

enables sharing selected objects in a database in your account with other Snowflake accounts Following objetcs can be shared

1. Tables
2. External Tables
3. Secure Views
4. Secure Materialized Views
5. Secure UDfs

- With secure data sharing, no actual data is copied or transferred between accounts. 
- All sharing is accomplished through snowflake's services layer and metadata store.
- The shared data doesn't take up any storage in consumer's account 
- the shared data doesn't contribute to consumer's monthly data storage charges
- Only charges are for compute resources used to query the shared data



# Shares

These are named Snowflake objects that captures all the information required to share a database.
Each share consists of 
- privleges that grant access to database and schemas containing the objects to share
- privileges that grant access to the specific objects in the database
- consumer account with which the database , schema and its objects are shared.

Shares are secure, configurable and contolled 100% by oprivider account

- new objects added to share become immediately available to all consumers, providing real time access to shared data
- access to a share can be revoked at any time

# Key Secure Data Sharing Features

- Zero Data Movement: Data is never copied or transfered. Consumers access the provider's data directly in real time
- live read only access : shred data is always up to date and consumers cannot modify the source data
- secure views & materialized vuews: providers can use secure views to hdie sensiitive data while still sharing releavnt insgihts
- Granular access control : uses stanrdar role based access control RBAC to manage permissions on speciifc databses, schemas and tables
- secure data exchange/ market place: enables sharing within a private, curated exchange or public marketplace for broader colaboration
- reader accounts: providers can create temporary, lightweight accounts for non snowflake customers to conume data with teh provider bearing all costs
- end to end encryotion: all data is encrypted in transit and at rest
- meatdata based: sharing happens though the service layer, meaning no storage costs for the conumer and only comput costs are beared.

# CLONING

powerful feature that allows you to create a clone of a database , schema or table without duplicating the underlying data. This means that the clone is created almost instantly and doesn't consume additional storage space initially, making it efficient way to manage and experiment with data.

Key features

1. Instant Cloning: Clones are created quickly, as they reference the same data as the original object. This allows for near instantanous creation of clones even for larger datasets
2. Storage efficiency. Initally, the clone oibject doesn;t take up additional storage as it points the same underlying data. as soon as changes are made to either the clone or the original object, storage gets consumed.
3. Independent changes: changes made to the clone or original object do not affect each other
4. Point in time cloning: you can create a clone of a table, schema or database as it existed at a speicific point in time using time travel features. This allows you to recover or analyse data from a previous state easily.

How to use Zero copy cloning

CREATE TABLE | DATABASE | SCHEMA MY_CLONED_TABLE CLONE MY_TABLE;

# TIME TRAVEL

It lets you access data that has been modified or deleted by quering previous versions of a table or a database at any point within a retention period of upto 90 days depedning on Snowflake editions.

1. Quering data using offset
synatx

select * from table at (offset =>-60*60*24)

2. Quering data using specific timestamp
syntax

select * from table before ( timestamp => '2025-01-02 14:21:50.51'::timestamp);

3. Querying data using queryid
syntax

select * from table before (statement =>'queryid');

USE CASES

1. Recovering data
2. Audit and compliance
3. Data Correctness
4. Business intelligence and analytics

Important points

1. Data Retention period : upto 90 days for enterprise and higeher
2. Cost: storage costs for historical data versions
3. Data modificaiton and time travel: time travel allows you to access data as it was at any point of time, but it wont be able to recover dropped objects like tables or schemas after the rention has passed

# FAIL SAFE

this feature needs upport from Snowflake support team to recover data after the retention period has passed and you get upto 7 extra days to recover the data

Why FAIL SAFE

Data is backed up at regular intervals, there can be a situation between the two backups during the data has been accidently deleted or changed .

Key features
1. failsafe period can be 0 or 7 days
2. For transient tables , it is 0 and for permanent it is 7
3. Users are not allowed to query on fail safe
4. data can be recovered only through snowflake support

# Accounts

- provider account: one who creats shares and makes them available to other snowflake accounts
- consumer account: one who creates a database from a share made availble by the data provider. As a conusmer, once you add a shared database into your account, you can access and query the objects in the database
- reader account: account created by the provider who can only consume data from the provider account that created it. It can query data but cannotb perform any DML

# Resharing
- requires snowflake data exchnage or direct sharing to a third party from the consumer side
- limitation: a consumer cannot directly reshare a shared database, they can create a table or view on top of the shared object and share with another account

# Direct Share
provides 
- secure
- real time
- simple method for sharing specific database objects between data provider and consumer account within same snowflake region.
- no data is actually copied or moved, eliminating deduplication and ensuring consumers always access live data.

Characterstics
1. Zero copy
2. Real time
3. Cost allocation
4. Access control: provider has all the access and revoke the access for any consumer
5. Region sepcific: direct shares are resticted within the same snowflake region
6. consumer type : sharing can occur with another existing snowflake accounts or with non snowflake user via a special provider managed reader account

Note: If you want to share data cross region, you need to use LISTING or REPLICATION

# Data Clean rooms

configurable isoloated snowflake env where collaborators can import data, specify what queries can be run againts that data and configure data protection setting ssuch as differnet privacy and specifying joinable and projectable columns. Access to clean room is by invitatiion only

Key concepts
1. Secure Collab
2. Provider and consumer roles
3. access control and policies
4. no code ui and developer api

COMMON USE CASE
party A and party B has their individual data but they want to share their common insights.

1. Advertising and marketing: measuring campaign performance without sharing user level data
2. Health care: accelerating drug research by enable secure data analysis bwteen labs and healthcare facilities wthout exposing patient info
3. Financial Services: improving fraud detection models and credit scoring while safegauridng customer data

# SNOWFLAKE MARKETPLACE

- explore and access listings : third party data, third party services
- provide and market listing : monetize data products, reach data consumers
- Snowflake Data cloud

# LISTINGS

enhanced method of data sharing that allows cross region sharing to share data, native apps or services with consumers.

Types

1. Private : these are secure, direct and controlled sharing with specific known snowflake accounts
- visibility : not visible to public
- access requirement : provider must know the specific org and acct name of the consumer
- use case : sharing with specific partners, clients or internal dept
- cabality: support free and paid models
- cross region : support cross cloud auto fulfillment to replicate data to the consumer's region automatically

2. Public : published on Snowflake marketplace, making them discoverable adn accesible to wider audience

- visibility : visible to all snowflake users
- access req : any snowflake user can find and initiate the get process for the listing
- use case : data monetization, publishing public data
- capabilities : supprts free trial (1-90 days) and paid listins
- approval process : requires approval from Snowflake to be published on the marketplace


# NATIVE APPS

enable developers to build, distribute and monetize data applications that run entirely within a consumer's snowflake account. This approach brings the application logic to where the data alrady resides, eliminating the need for data movement and enhancing security and governance

Key feature

- in account execution: native apps are installed and ru directly in the consumer's snowfalke acct
- monetization: providers can list their applications on snowflake marketplace using licencing and pricing models
- development tools: applications can be built using familar language and tools within snowflake ecosystme
-intellectual property protection : the framework is designed to protect the provider's underlying code and business logic
- snowpark container service: for complex apps, native apps can leverage SPCS to run containerized workloads allowing integraton of AI/ML models

Workflow

DEVELOP -> PUBLISH -> INSTALL -> USE
